# List Vs Tuples

* Lists and tuples are types of arrays (ordered data structures)
* An array stores data in a fixed order, where position matters
* Knowing the position lets you access elements in O(1) time
* Arrays can be implemented in different ways, with different trade-offs


Python provides two main types:
* Lists: dynamic, can grow, shrink, and be modified
* Tuples: static, fixed size, and immutable

_Bonus notes on notations_

O(1) means the operation takes the same amount of time no matter how large your data is.

The "O" comes from Big O notation - a way to describe how fast an operation is as data grows.

| Notation | Name | Example | Meaning |
|---|---|---|---|
| O(1) | Constant | `list[3]` | Same time regardless of size |
| O(n) | Linear | Searching unsorted list | Doubles when data doubles |
| O(log n) | Logarithmic | Binary search | Grows slowly as data grows |
| O(n²) | Quadratic | Nested loops | Gets very slow very fast |



In [4]:
import time

big_list = list(range(100_000_000))  # 100 million elements

# O(1) — instant, regardless of list size
start = time.perf_counter()
_ = big_list[99_999_999]   # Jump straight to last element
print(f"O(1) scan time perf ",time.perf_counter() - start)

# O(n) — scans every element
start = time.perf_counter()
_ = 99_999_999 in big_list  # Has to check each one
print(f"O(n) scan time perf " ,time.perf_counter() - start)

O(1) scan time perf  8.425000123679638e-05
O(n) scan time perf  1.6471223750850186


# Searching Lists More Efficiently

## The Problem with Normal Search
Searching through an unsorted list checks **every element one by one** — this is O(n) and gets slow fast.

The solution? **Sort first, then use binary search.**

## Step 1: Sort the List
Python lists have a built-in sort using **Tim Sort**:
- Best case: O(n)
- Worst case: O(n log n)

It's smart — it looks at your data and picks the best sorting strategy automatically.

## Step 2: Binary Search
Instead of checking every element, binary search **cuts the list in half** each time:

1. Look at the **middle** element
2. Too big? → Search the **left half**
3. Too small? → Search the **right half**
4. Repeat until found

**Result: O(log n)** — much faster than scanning everything


In [ ]:
def binary_search(needle, haystack):
    imin, imax = 0, len(haystack)
    while True:
        if imin > imax:
            return -1
        midpoint = (imin + imax) // 2
        if haystack[midpoint] > needle:
            imax = midpoint
        elif haystack[midpoint] < needle:
            imin = midpoint + 1
        else:
            return midpoint

## The `bisect` Module — Easier Binary Search
Python's built-in `bisect` handles all of this for you:
- **`bisect.insort()`** — inserts a new element in the correct sorted position
- **`bisect.bisect_left()`** — finds where an element is (or should be)

In [ ]:
import bisect

numbers = []
bisect.insort(numbers, 50)   # List stays sorted automatically
bisect.insort(numbers, 20)
bisect.insort(numbers, 80)

print(numbers)  # [20, 50, 80] ← always sorted!

## Dictionary vs Binary Search — Which to use?

| Approach | Speed | When to use |
|---|---|---|
| Dictionary lookup | O(1) | When you need very frequent lookups |
| Binary search | O(log n) | When data is already sorted |
| Convert to dict first | O(n) + O(1) | Conversion cost may not be worth it |


## Golden Rule
> **Pick the right data structure and stick with it.**
> Converting between structures has a cost — factor that in before switching.

# When to Use Each Approach in Data Engineering


## 1. Linear Search — O(n)
**"Check every record one by one"**

Use when:
- Dataset is **small** (< 1,000 rows)
- Data is **unsorted** and you only search once
- You need to apply a **filter condition** across all rows

```python
# Find all failed pipeline runs
failed_jobs = [job for job in pipeline_logs if job['status'] == 'failed']
```

Avoid when:
- Dataset is large — it will get very slow



## 2. Binary Search — O(log n)
**"Split and search a sorted list"**

Use when:
- Data is **already sorted** (timestamps, IDs, dates)
- You need to find **closest match** (fuzzy lookups)
- You're comparing **two similar datasets**

```python
import bisect

# Find the closest timestamp to a given event time
timestamps = sorted([...])  # Already sorted log timestamps
idx = bisect.bisect_left(timestamps, target_time)
# Great for aligning two event streams that aren't identical
```

Avoid when:
- Data changes frequently (re-sorting is expensive)
- Data is completely unsorted (sort cost may not be worth it)


## 3. Dictionary Lookup — O(1)
**"Jump directly to what you need"**

Use when:
- You do **repeated lookups** on the same dataset
- Joining or mapping data (like a lookup table)
- Deduplication — checking if a record already exists

```python
# Map customer_id → customer name (lookup table)
customer_map = {c['id']: c['name'] for c in customers}

# Now enrich millions of orders instantly
for order in orders:
    order['customer_name'] = customer_map[order['customer_id']]  # O(1) each time
```

Avoid when:
- You only search **once** (converting to dict costs O(n))
- Keys are not unique (dicts don't allow duplicate keys)


## 4. Sorting + Binary Search (bisect) — O(n log n) + O(log n)
**"Sort once, search many times cheaply"**

Use when:
- Data arrives in a **stream** and must stay sorted
- You need **range queries** (find all records between date A and B)
- You're doing **time-series alignment**

```python
import bisect

event_times = []

# As new events arrive, insert them in sorted order
bisect.insort(event_times, new_event_timestamp)

# Find all events in a time window — very fast
start = bisect.bisect_left(event_times, window_start)
end   = bisect.bisect_right(event_times, window_end)
window_events = event_times[start:end]
```


## Quick Decision Guide

| Situation | Best Approach |
|---|---|
| Small dataset, one-off search | Linear Search O(n) |
| Already sorted data (timestamps, IDs) | Binary Search O(log n) |
| Repeated lookups, joins, mapping | Dictionary O(1) |
| Streaming data that must stay sorted | bisect O(log n) |
| Deduplication of large datasets | Dictionary / Set O(1) |
| Range queries on time-series data | Sort + bisect |

---

## Real Pipeline Example

```python
# You have 10 million orders and a customer lookup table

customers = [{'id': 1, 'name': 'Alice'}, ...]  # 50,000 customers

# BAD — Linear search inside a loop = O(n × m) — extremely slow
for order in orders:
    customer = next(c for c in customers if c['id'] == order['customer_id'])

# GOOD — Build dict once O(n), then look up in O(1) each time
customer_map = {c['id']: c['name'] for c in customers}
for order in orders:
    order['customer_name'] = customer_map[order['customer_id']]
```

> **Rule of thumb:** If you're searching inside a loop — use a dictionary.
> If your data is naturally ordered — use binary search.